# 🚀 Video Upscaler AI — Kaggle GPU Backend

شغّل ده على Kaggle بـ **GPU T4 x2** (Settings → Accelerator → GPU T4 x2) وهيبقى سيرفر Real-ESRGAN قوي جداً متصل بتطبيقك تلقائياً.

**الخطوات:**
1. Settings (يمين) → Accelerator: **GPU T4 x2** → Internet: **On**
2. Add-ons → Secrets → أضف `ADMIN_KEY` (من ملف cloud_admin_key.txt) و `WORKER_URL` = `https://upscaler-cloud.cracknew37.workers.dev`
3. Run All ▶▶ — بعد دقيقتين هتشوف `✅ REGISTERED` والتطبيق هيستخدم Kaggle أوتوماتيك
4. سيبه شغال (Kaggle يسمح 12 ساعة متواصلة، 30 ساعة/أسبوع)

> لما تقفل النوتبوك، الـ Worker يرجع تلقائياً لـ HuggingFace ZeroGPU.

In [ ]:
# 1) Install
!pip install -q gradio==4.44.1 realesrgan basicsr facexlib gfpgan opencv-python-headless 2>&1 | tail -1
# basicsr compat with new torchvision
import glob, re
for f in glob.glob('/usr/local/lib/python3*/dist-packages/basicsr/data/degradations.py') + glob.glob('/opt/conda/lib/python3*/site-packages/basicsr/data/degradations.py'):
    s = open(f).read().replace('from torchvision.transforms.functional_tensor import rgb_to_grayscale', 'from torchvision.transforms.functional import rgb_to_grayscale')
    open(f, 'w').write(s)
import torch; print('GPU:', torch.cuda.get_device_name(0), '| count:', torch.cuda.device_count())

In [ ]:
# 2) Models (all 4 variants the app knows about)
import os, urllib.request
os.makedirs('weights', exist_ok=True)
W = {
 'general':  'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth',
 'wdn':      'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-wdn-x4v3.pth',
 'anime':    'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth',
 'x4plus':   'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
}
for k, u in W.items():
    p = f'weights/{k}.pth'
    if not os.path.exists(p): urllib.request.urlretrieve(u, p); print('downloaded', k)
print('models ready')

In [ ]:
# 3) Inference server (Gradio → same API shape the Worker already speaks: fn 0 = image, fn 1 = video)
import gradio as gr, cv2, numpy as np, tempfile, subprocess, torch
from PIL import Image
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan.archs.srvgg_arch import SRVGGNetCompact
from realesrgan import RealESRGANer

def build(kind):
    if kind == 'x4plus':
        net = RRDBNet(3, 3, 64, 23, 32, 4)
    elif kind == 'anime':
        net = SRVGGNetCompact(3, 3, 16, 16, 4, 'prelu')
    else:
        net = SRVGGNetCompact(3, 3, 64, 32, 4, 'prelu')
    return RealESRGANer(scale=4, model_path=f'weights/{kind}.pth', model=net, tile=0, half=True, gpu_id=0)

UPS = {k: build(k) for k in ['general', 'wdn', 'anime', 'x4plus']}
print('loaded', list(UPS))

def infer_image(img: Image.Image, scale: int, model: str = 'general'):
    up = UPS.get(model, UPS['general'])
    bgr = cv2.cvtColor(np.array(img.convert('RGB')), cv2.COLOR_RGB2BGR)
    out, _ = up.enhance(bgr, outscale=scale)
    return Image.fromarray(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))

def infer_video(path: str, scale: int, model: str = 'general', progress=gr.Progress()):
    up = UPS.get(model, UPS['general'])
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) * scale; h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) * scale
    raw = tempfile.mktemp(suffix='.mp4'); final = tempfile.mktemp(suffix='.mp4')
    # pipe frames straight into ffmpeg h264 (NVENC if available) for quality + speed
    codec = 'h264_nvenc' if subprocess.run(['ffmpeg','-hide_banner','-encoders'],capture_output=True,text=True).stdout.find('h264_nvenc')>=0 else 'libx264'
    ff = subprocess.Popen(['ffmpeg','-y','-loglevel','error','-f','rawvideo','-pix_fmt','bgr24','-s',f'{w}x{h}','-r',str(fps),'-i','-',
                           '-c:v',codec,'-preset','p4' if codec=='h264_nvenc' else 'fast','-b:v',f'{int(w*h*fps*0.09/1000)}k','-pix_fmt','yuv420p', raw], stdin=subprocess.PIPE)
    prev = None; prev_out = None; i = 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        # duplicate-frame skip (same trick as the phone app)
        small = cv2.resize(frame, (32, 32), interpolation=cv2.INTER_AREA).astype(np.int16)
        if prev is not None and np.abs(small - prev).mean() < 1.2 and prev_out is not None:
            out = prev_out
        else:
            out, _ = up.enhance(frame, outscale=scale); prev_out = out
        prev = small
        ff.stdin.write(out.tobytes()); i += 1
        if i % 10 == 0: progress(i / max(n, 1), desc=f'{i}/{n}')
    ff.stdin.close(); ff.wait(); cap.release()
    # mux original audio back
    r = subprocess.run(['ffmpeg','-y','-loglevel','error','-i',raw,'-i',path,'-c:v','copy','-c:a','aac','-map','0:v:0','-map','1:a:0?','-shortest',final])
    return final if r.returncode == 0 else raw

MODELS = ['general', 'wdn', 'anime', 'x4plus']
with gr.Blocks() as demo:
    gr.Markdown('# Video Upscaler AI — Kaggle GPU backend')
    with gr.Tab('Image'):
        ii = gr.Image(type='pil'); si = gr.Radio([2,4,8], value=4, type='value', label='scale'); mi = gr.Dropdown(MODELS, value='general', label='model'); oi = gr.Image(type='pil')
        gr.Button('Run').click(infer_image, [ii, si, mi], oi, api_name='image')
    with gr.Tab('Video'):
        iv = gr.Video(); sv = gr.Radio([2,4], value=4, type='value', label='scale'); mv = gr.Dropdown(MODELS, value='general', label='model'); ov = gr.Video()
        gr.Button('Run').click(infer_video, [iv, sv, mv], ov, api_name='video')
demo.queue(max_size=20)
app, local_url, share_url = demo.launch(share=True, prevent_thread_lock=True, show_error=True)
print('PUBLIC URL:', share_url)

In [ ]:
# 4) Register with the Cloudflare Worker so the app uses this GPU automatically (HF stays as fallback)
import requests, os
try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient(); ADMIN = sec.get_secret('ADMIN_KEY'); WORKER = sec.get_secret('WORKER_URL')
except Exception:
    ADMIN = os.environ.get('ADMIN_KEY', ''); WORKER = os.environ.get('WORKER_URL', 'https://upscaler-cloud.cracknew37.workers.dev')
host = share_url.rstrip('/')
# format: host|fn_image|fn_video ; Kaggle first, then HF fallback
backends = f'{host}|0|1|3,https://nick088-real-esrgan-pytorch.hf.space|0|3|2'
r = requests.post(WORKER + '/backends', data=backends, headers={'X-Admin-Key': ADMIN}, timeout=30)
print('✅ REGISTERED' if r.ok else '❌ register failed', r.status_code, r.text[:200])
print('Health:', requests.get(WORKER + '/health', timeout=30).json())

In [ ]:
# 5) Keep alive + auto-unregister when the session ends (so the Worker falls back to HF)
import time, atexit
def unregister():
    try: requests.post(WORKER + '/backends', data='https://nick088-real-esrgan-pytorch.hf.space|0|3|2', headers={'X-Admin-Key': ADMIN}, timeout=15); print('unregistered')
    except Exception as e: print(e)
atexit.register(unregister)
print('Serving… leave this cell running. GPU:', torch.cuda.get_device_name(0))
while True:
    time.sleep(300)
    try: requests.get(host + '/queue/status', timeout=10)
    except Exception as e: print('ping', e)